<a href="https://colab.research.google.com/github/RMRL/-/blob/master/%EB%8D%94%20%EB%A9%8B%EC%A7%84%20%EB%B2%88%EC%97%AD%EA%B8%B0%20%EB%A7%8C%EB%93%A4%EA%B8%B0%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🇰🇷→🇺🇸 더 멋진 번역기 - 완벽 통합판
# ============================================================

import os, re, math, urllib.request, tarfile
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

print("=" * 70)
print("  🇰🇷→🇺🇸 더 멋진 번역기 - 전체 파이프라인")
print("=" * 70)

# ============================================================
# 1. 설정
# ============================================================
DATA_DIR = os.path.expanduser("~/work/transformer/data")
os.makedirs(DATA_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n[장치] {DEVICE}")

# ============================================================
# 2. 데이터 준비
# ============================================================
print("\n[1/7] 데이터 준비 중...")

kor_file = os.path.join(DATA_DIR, "korean-english-park.train.ko")
eng_file = os.path.join(DATA_DIR, "korean-english-park.train.en")
tar_path = os.path.join(DATA_DIR, "korean-english-park.train.tar.gz")

# 원본 데이터가 없으면 다운로드
if not os.path.exists(kor_file) or not os.path.exists(eng_file):
    url = "https://github.com/jungyeul/korean-parallel-corpora/raw/master/korean-english-news-v1/korean-english-park.train.tar.gz"

    if not os.path.exists(tar_path):
        print("   GitHub에서 다운로드 중...")
        try:
            urllib.request.urlretrieve(url, tar_path)
            print("   다운로드 완료!")
        except:
            print("   다운로드 실패!")

    # 압축 해제
    if os.path.exists(tar_path):
        print("   압축 해제 중...")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=DATA_DIR)
        print("   압축 해제 완료!")

# 원본 데이터 로드 (없으면 샘플 생성)
if os.path.exists(kor_file) and os.path.exists(eng_file):
    print("   원본 데이터 로드 중...")
    with open(kor_file, "r", encoding="utf-8") as f:
        kor_lines = [l.strip() for l in f if l.strip()][:100000]
    with open(eng_file, "r", encoding="utf-8") as f:
        eng_lines = [l.strip() for l in f if l.strip()][:100000]
else:
    print("   샘플 데이터 생성 중...")
    ko_temps = [
        "오바마는 미국의 대통령이다.", "시민들은 도시 속에서 살고 있다.",
        "커피가 전혀 필요하지 않다.", "일곱 명의 사망자가 발생했다.",
        "나는 대학생입니다.", "오늘 날씨가 매우 좋다.",
        "그는 병원에서 일하는 의사이다.", "이 책은 정말 재미있다.",
        "우리는 오랜 친구 사이이다.", "그녀는 고등학교 선생님이다.",
        "컴퓨터가 매우 빠르게 작동한다.", "규칙적인 운동은 건강에 좋다.",
        "음악은 내 삶의 일부이다.", "저녁으로 무엇을 먹을까?",
        "내일 아침에 비가 올 것이다.", "대한민국은 아름다운 나라다.",
        "수학은 가장 어려운 과목이다.", "영화를 함께 보러 가자.",
        "고양이가 매우 귀엽다.", "시간이 정말 빨리 흐른다.",
        "회사가 새로운 제품을 출시했다.", "학생들이 도서관에서 공부한다.",
        "우리는 주말에 여행을 갈 계획이다.", "커피 한 잔 주세요.",
        "그는 노래를 매우 잘 부른다.", "이 문제는 생각보다 어렵다.",
        "새로운 언어를 배우는 것은 재미있다.", "그녀는 아름다운 목소리를 가졌다.",
        "나는 최고의 식당을 알고 있다.", "우리는 환경을 보호해야 한다.",
        "그 작가는 많은 책을 썼다.", "오늘따라 기분이 매우 좋다.",
        "이 기술은 미래에 중요하다.", "그들은 서로를 깊이 사랑한다.",
        "운동은 스트레스 해소에 도움이 된다.", "저녁 하늘이 정말 아름답다.",
        "우리 팀이 대회에서 우승했다.", "따뜻한 차가 마시고 싶다.",
        "영화는 실화를 바탕으로 제작되었다.", "인공지능 기술이 빠르게 발전한다.",
        "오늘은 일찍 퇴근하고 싶다.", "이 음식은 정말 맛있다.",
        "그의 연설이 모두를 감동시켰다.", "매일 성장하는 것이 중요하다.",
        "가족과 함께하는 시간은 소중하다.", "겨울이 다가오고 있다.",
        "좋은 책은 인생을 바꿀 수 있다.", "산책하며 음악 듣는 것을 좋아한다.",
        "그 도시는 밤에 더 아름답다.", "친구들과의 여행은 항상 즐겁다.",
    ]
    en_temps = [
        "obama is the president of the united states.", "citizens are living in the city.",
        "coffee is not needed at all.", "seven deaths occurred.",
        "i am a college student.", "the weather is very nice today.",
        "he is a doctor working at a hospital.", "this book is really interesting.",
        "we are old friends.", "she is a high school teacher.",
        "the computer runs very fast.", "regular exercise is good for health.",
        "music is a part of my life.", "what should we eat for dinner?",
        "it will rain tomorrow morning.", "south korea is a beautiful country.",
        "mathematics is the most difficult subject.", "let's go watch a movie together.",
        "the cat is very cute.", "time passes really quickly.",
        "the company launched a new product.", "students are studying in the library.",
        "we plan to go on a trip this weekend.", "one cup of coffee please.",
        "he sings very well.", "this problem is harder than expected.",
        "learning a new language is fun.", "she has a beautiful voice.",
        "i know the best restaurant.", "we must protect the environment.",
        "the author wrote many books.", "i feel very good today.",
        "this technology is important for the future.", "they love each other deeply.",
        "exercise helps relieve stress.", "the evening sky is really beautiful.",
        "our team won the competition.", "i want to drink warm tea.",
        "the movie was based on a true story.", "artificial intelligence is developing rapidly.",
        "i want to leave work early today.", "this food is really delicious.",
        "his speech moved everyone.", "it is important to grow every day.",
        "time with family is precious.", "winter is approaching.",
        "a good book can change a life.", "i like walking while listening to music.",
        "the city is more beautiful at night.", "traveling with friends is always fun.",
    ]
    kor_lines = ko_temps * 2000
    eng_lines = en_temps * 2000

n = min(len(kor_lines), len(eng_lines))
kor_lines, eng_lines = kor_lines[:n], eng_lines[:n]
print(f"   총 {n:,}개의 문장 쌍")

# ============================================================
# 3. 전처리
# ============================================================
print("\n[2/7] 전처리 중...")

def preprocess(text, lang="ko"):
    if lang == "en":
        text = text.lower()
    text = re.sub(r"[^a-zA-Z가-힣0-9\s?.!,;:'\"()\[\]-]", " ", text)
    text = re.sub(r'([?.!,;:\'\"()\[\]{}])', r' \1 ', text)
    return re.sub(r'\s+', ' ', text).strip()

kor_clean = [preprocess(k, "ko") for k in kor_lines]
eng_clean = [preprocess(e, "en") for e in eng_lines]

# 파일 저장
with open(os.path.join(DATA_DIR, "kor_clean.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(kor_clean))
with open(os.path.join(DATA_DIR, "eng_clean.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(eng_clean))

print(f"   전처리 완료: {len(kor_clean):,}쌍")

# ============================================================
# 4. 토크나이저
# ============================================================
print("\n[3/7] 토크나이저 학습 중...")

unique_chars = len(set("".join(kor_clean)))
vocab_size = min(8000, max(200, unique_chars + 100))
print(f"   사전 크기: {vocab_size}")

spm.SentencePieceTrainer.Train(
    f"--input={DATA_DIR}/kor_clean.txt "
    f"--model_prefix={DATA_DIR}/ko_model "
    f"--vocab_size={vocab_size} "
    f"--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 "
    f"--pad_piece=<PAD> --bos_piece=<BOS> --eos_piece=<EOS> --unk_piece=<UNK> "
    f"--model_type=bpe --character_coverage=0.9995"
)

spm.SentencePieceTrainer.Train(
    f"--input={DATA_DIR}/eng_clean.txt "
    f"--model_prefix={DATA_DIR}/en_model "
    f"--vocab_size={vocab_size} "
    f"--pad_id=0 --bos_id=1 --eos_id=2 --unk_id=3 "
    f"--pad_piece=<PAD> --bos_piece=<BOS> --eos_piece=<EOS> --unk_piece=<UNK> "
    f"--model_type=bpe --character_coverage=1.0"
)

ko_tok = spm.SentencePieceProcessor()
ko_tok.Load(f"{DATA_DIR}/ko_model.model")

en_tok = spm.SentencePieceProcessor()
en_tok.Load(f"{DATA_DIR}/en_model.model")
en_tok.SetEncodeExtraOptions("bos:eos")

SRC_VOCAB = ko_tok.GetPieceSize()
TGT_VOCAB = en_tok.GetPieceSize()
print(f"   완료! src={SRC_VOCAB}, tgt={TGT_VOCAB}")

# ============================================================
# 5. 학습 데이터
# ============================================================
print("\n[4/7] 학습 데이터 준비 중...")

enc_in, dec_in, dec_out = [], [], []

for ko, en in tqdm(zip(kor_clean, eng_clean), total=len(kor_clean)):
    k_ids = ko_tok.EncodeAsIds(ko)
    e_ids = en_tok.EncodeAsIds(en)
    if len(k_ids) <= 50 and len(e_ids) <= 50:
        enc_in.append(torch.tensor(k_ids, dtype=torch.long))
        dec_in.append(torch.tensor(e_ids[:-1], dtype=torch.long))
        dec_out.append(torch.tensor(e_ids[1:], dtype=torch.long))

enc_train = torch.nn.utils.rnn.pad_sequence(enc_in, batch_first=True, padding_value=0)
dec_in_train = torch.nn.utils.rnn.pad_sequence(dec_in, batch_first=True, padding_value=0)
dec_out_train = torch.nn.utils.rnn.pad_sequence(dec_out, batch_first=True, padding_value=0)

n_data = enc_train.shape[0]
print(f"   학습 데이터: {n_data:,}문장")
print(f"   enc: {enc_train.shape}, dec_in: {dec_in_train.shape}")

# ============================================================
# 6. 모델
# ============================================================
print("\n[5/7] 모델 생성 중...")

class Translator(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=4, n_layers=3, d_ff=512, max_len=100):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

        self.transformer = nn.Transformer(
            d_model, n_heads, n_layers, n_layers, d_ff, dropout=0.1, batch_first=True
        )
        self.output = nn.Linear(d_model, tgt_vocab)
        self.output.weight = self.tgt_emb.weight

    def forward(self, src, tgt):
        src_pad, tgt_pad = (src == 0), (tgt == 0)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1]).to(src.device)

        src_emb = self.src_emb(src) * math.sqrt(self.d_model) + self.pe[:src.shape[1], :]
        tgt_emb = self.tgt_emb(tgt) * math.sqrt(self.d_model) + self.pe[:tgt.shape[1], :]

        out = self.transformer(src_emb, tgt_emb,
                               src_key_padding_mask=src_pad,
                               tgt_mask=tgt_mask,
                               tgt_key_padding_mask=tgt_pad)
        return self.output(out)

    @torch.no_grad()
    def translate(self, text, max_len=50):
        self.eval()
        src = torch.tensor([ko_tok.EncodeAsIds(text)], dtype=torch.long).to(DEVICE)

        src_pad = (src == 0)
        src_emb = self.src_emb(src) * math.sqrt(self.d_model) + self.pe[:src.shape[1], :]
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_pad)

        ids = [en_tok.bos_id()]
        for _ in range(max_len):
            tgt = torch.tensor([ids], dtype=torch.long).to(DEVICE)
            tgt_pad = (tgt == 0)
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.shape[1]).to(DEVICE)

            tgt_emb = self.tgt_emb(tgt) * math.sqrt(self.d_model) + self.pe[:tgt.shape[1], :]
            out = self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_pad)

            nxt = self.output(out[:, -1, :]).argmax(dim=-1).item()
            ids.append(nxt)
            if nxt == en_tok.eos_id():
                break

        return en_tok.DecodeIds(ids[1:-1]) if len(ids) > 2 else en_tok.DecodeIds(ids[1:])

model = Translator(SRC_VOCAB, TGT_VOCAB).to(DEVICE)
print(f"   파라미터: {sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 7. 학습
# ============================================================
print("\n[6/7] 학습 시작!")
print("=" * 50)

enc_train = enc_train.to(DEVICE)
dec_in_train = dec_in_train.to(DEVICE)
dec_out_train = dec_out_train.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98))
criterion = nn.CrossEntropyLoss(ignore_index=0)

BATCH, EPOCHS = 64, 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    idx = torch.randperm(n_data)

    for i in range(0, n_data, BATCH):
        bi = idx[i:min(i+BATCH, n_data)]

        optimizer.zero_grad()
        pred = model(enc_train[bi], dec_in_train[bi])
        loss = criterion(pred.reshape(-1, pred.shape[-1]), dec_out_train[bi].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        print(f"   Epoch {epoch+1:2d}/{EPOCHS} | Loss: {total_loss/(n_data/BATCH):.4f}")

print("   학습 완료!")

# ============================================================
# 8. 저장 + 테스트
# ============================================================
torch.save(model.state_dict(), os.path.join(DATA_DIR, "translator.pt"))
print(f"\n   모델 저장: {DATA_DIR}/translator.pt")

print("\n" + "=" * 70)
print("[7/7] 번역 결과")
print("=" * 70)

tests = [
    "오바마는 대통령이다.",
    "시민들은 도시 속에 산다.",
    "커피는 필요 없다.",
    "일곱 명의 사망자가 발생했다.",
    "오늘 날씨가 좋다.",
    "나는 학생입니다.",
    "그는 의사입니다.",
    "커피 한 잔 주세요.",
    "새로운 언어를 배우는 것은 재미있다.",
    "운동은 건강에 좋다.",
    "이 책은 정말로 재미있다.",
    "영화를 함께 보러 가자.",
]

for i, sent in enumerate(tests, 1):
    result = model.translate(preprocess(sent, "ko"))
    print(f"\n{i:2d}. 🇰🇷 {sent}")
    print(f"    🇺🇸 {result}")

print("\n" + "=" * 70)
print("🎉 더 멋진 번역기 완성!")
print("=" * 70)

# 대화형 모드
print("\n💬 대화형 번역 (종료: q)")
while True:
    u = input("\n🇰🇷 ").strip()
    if u.lower() == 'q':
        print("👋 안녕!")
        break
    if u:
        print(f"🇺🇸 {model.translate(preprocess(u, 'ko'))}")

  🇰🇷→🇺🇸 더 멋진 번역기 - 전체 파이프라인

[장치] cpu

[1/7] 데이터 준비 중...
   GitHub에서 다운로드 중...
   다운로드 완료!
   압축 해제 중...


/tmp/ipykernel_6783/1259323802.py:50: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATA_DIR)


   압축 해제 완료!
   원본 데이터 로드 중...
   총 94,123개의 문장 쌍

[2/7] 전처리 중...
   전처리 완료: 94,123쌍

[3/7] 토크나이저 학습 중...
   사전 크기: 1814
   완료! src=1814, tgt=1814

[4/7] 학습 데이터 준비 중...


  0%|          | 0/94123 [00:00<?, ?it/s]

   학습 데이터: 57,439문장
   enc: torch.Size([57439, 50]), dec_in: torch.Size([57439, 49])

[5/7] 모델 생성 중...
   파라미터: 4,885,270

[6/7] 학습 시작!


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
